In [1]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

In [2]:
load_dotenv(Path.cwd().parent / ".env")

False

In [3]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [4]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

def get_meta_data(endpoint, params=None, headers=None):
    url = f"{URL_META_DATA}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    return response

## Save API info as JSON

Save all endpoints needed for each company

In [5]:
import json

def save_json(company_num):
    
    endpoints = {
        "profile":   get(f"/company/{company_num}"),
        "charges":   get(f"/company/{company_num}/charges"),
        "officers":  get(f"/company/{company_num}/officers"),
        "filing":    get(f"/company/{company_num}/filing-history"),
        'PSC':       get(f'/company/{company_num}/persons-with-significant-control')
    }

    for name, data in endpoints.items():
        with open(f"company_info_json/{company_num}_{name}.json", "w") as f:
            json.dump(data, f)

## Create a dictionary of the company info from JSON

In [6]:
def test_variables(company_num):
    endpoints = ['profile', 'charges', 'officers', 'filing','PSC']
    test_dict = {}
    for name in endpoints:
        with open(f"company_info_json/{company_num}_{name}.json", "r") as f:
            test_dict[name] = json.load(f)
    return test_dict


In [7]:
company_number = '11257129'
save_json(company_number) # 200 means successfull

200
200
200
200
200


In [8]:
test_variables(company_number)['PSC']

{'items_per_page': 25,
 'items': [{'etag': '1526fb8cef6f17bd3fb9ad429fa7561a1b583c84',
   'notified_on': '2025-12-12',
   'name': 'Millish Holdings Limited',
   'links': {'self': '/company/11257129/persons-with-significant-control/corporate-entity/xbkwcZWM0gIROdg506Ylc5oz6MU'},
   'identification': {'legal_form': 'Limited By Shares',
    'legal_authority': 'Isle Of Man',
    'country_registered': 'Isle Of Man',
    'place_registered': 'Isle Of Man Companies Register',
    'registration_number': '017811v'},
   'ceased': False,
   'kind': 'corporate-entity-person-with-significant-control',
   'address': {'address_line_1': "20 St Mary's Court",
    'address_line_2': 'Hill Street',
    'country': 'Isle Of Man',
    'locality': 'Douglas',
    'premises': '2nd Floor',
    'region': 'Im1 1eu'},
   'natures_of_control': ['ownership-of-shares-75-to-100-percent',
    'voting-rights-75-to-100-percent',
    'right-to-appoint-and-remove-directors']},
  {'etag': '050edfd9a9198d6d647b847a13baa2ecb2cd

## Pull out variables needed

List of variables needed:

In [8]:
variables_profile = ['company_name', 'date_of_creation', 'sic_codes', 'type', 'jurisdiction', 'has_charges','has_insolvency_history']
variables_profile_acc = ['type']
variables_charges = ['status']



Compile them into a dictionary of the company

In [ ]:
company_info = {}

# Compile company profile data
for i in variables_profile:
    company_info[i] = test_variables(company_number)['profile'][i]

# Compile locality seperately
company_info['locality'] = test_variables(company_number)['profile']['registered_office_address']['locality'] 

# Compile account type
company_info['account_type'] = test_variables(company_number)['profile']['accounts']['last_accounts']['type']

# Compile Charges (some may not have)
try:
    for i in variables_charges:
        company_info[i] = test_variables(company_number)['charges']['items'][0][i]
except IndexError:
    print(f'Skipping charges info of {test_variables(company_number)['profile']['company_name']} - list index out of range')

# Compile persons entitled
try:
    persons = test_variables(company_number)['charges']['items'][0]['persons_entitled']
    company_info['persons_entitled'] = [p['name'] for p in persons]
except IndexError:
    print(f'Skipping person entitled of {test_variables(company_number)["profile"]["company_name"]} - list index out of range')


  

In [10]:
test_variables(company_number)['profile']['accounts']['next_accounts']

{'due_on': '2026-12-31',
 'overdue': False,
 'period_end_on': '2026-03-31',
 'period_start_on': '2025-04-01'}

In [11]:
company_info # has_charges is not working (False but have outstanding charges in the charges field)

{'company_name': 'FELLTEN LTD',
 'date_of_creation': '2018-03-15',
 'sic_codes': ['29310', '29320'],
 'type': 'ltd',
 'jurisdiction': 'england-wales',
 'has_charges': False,
 'has_insolvency_history': False,
 'locality': 'Bristol',
 'account_type': 'total-exemption-full',
 'status': 'outstanding',
 'persons_entitled': ['Hsbc UK Bank PLC']}

In [12]:
pip install lxml


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Pulling Account iXBRL or PDF

In [13]:
from bs4 import BeautifulSoup
import lxml

# 1. Find latest ACCOUNTS filing.
#    The category filter should already restrict this, but we re-filter defensively:
#    if a non-accounts filing (e.g. a confirmation statement) is returned, its PDF
#    is just a Statement of Capital / shareholder list -- no balance sheet or debt.
fh = get(f"/company/{company_number}/filing-history", params={"category": "accounts"})
accounts_items = [it for it in fh["items"] if it.get("category") == "accounts"]
if not accounts_items:
    raise ValueError(f"No accounts filings found for {company_number}")

item = accounts_items[0]
print(f"Using accounts filing: type={item.get('type')} | "
      f"date={item.get('date')} | {item.get('description')}")

# Hard-stop: AA is the Companies House form type for annual accounts.
# Anything else means we're about to save the wrong document.
assert item.get("type") == "AA", f"Expected accounts (AA) filing, got {item.get('type')!r}"

doc_meta_url = item["links"]["document_metadata"]
doc_id = doc_meta_url.rstrip("/").split("/")[-1]

# 2. Check what formats are available for this document
meta = get_meta_data(doc_id)
resources = meta.json().get("resources", {})
print("Available formats:", list(resources.keys()))

# 3a. iXBRL available — parse XBRL-tagged balance sheet figures
if "application/xhtml+xml" in resources:
    content = get_meta_data(f"{doc_id}/content", headers={"Accept": "application/xhtml+xml"})
    # Use lxml-xml parser: handles XML namespaces correctly (required for ix: tags)
    soup = BeautifulSoup(content.text, "lxml-xml")
    wanted = {"NetAssetsLiabilities", "Equity", "CashBankOnHand",
              "Debtors", "FixedAssets", "CurrentAssets"}
    for tag in soup.find_all(["ix:nonfraction", "ix:nonFraction"]):
        name = (tag.get("name") or "").split(":")[-1]
        if name in wanted:
            print(name, tag.get("contextref"), tag.text.strip())

# 3b. PDF fallback — save to disk
elif "application/pdf" in resources:
    content = get_meta_data(f"{doc_id}/content", headers={"Accept": "application/pdf"})
    pdf_path = f"company_info_json/{company_number}_accounts.pdf"
    with open(pdf_path, "wb") as f:
        f.write(content.content)
    print(f"iXBRL not available — accounts PDF ({item.get('date')}) saved to {pdf_path}")

else:
    print("No iXBRL or PDF available. Formats found:", list(resources.keys()))


200
Using accounts filing: type=AA | date=2026-02-03 | accounts-with-accounts-type-total-exemption-full
Available formats: ['application/pdf', 'application/xhtml+xml']
FixedAssets None 1,458,130
FixedAssets None 1,055,429
Debtors None 460,995
Debtors None 401,662
CashBankOnHand None 59,684
CashBankOnHand None 110,139
CurrentAssets None 1,749,805
CurrentAssets None 1,685,081
NetAssetsLiabilities None 452,679
NetAssetsLiabilities None 622,245
Equity None 3
Equity None 3
Equity None 500,000
Equity None 500,000
Equity None 47,324
Equity None 122,242
Equity None 452,679
Equity None 622,245
Debtors None 460,995
Debtors None 401,662
